In [ ]:
'''

IMPORT MODEL ARCHITECTURE and DATA GENERATOR

'''

In [ ]:
MODEL_PATH = "window_attention_transformer_final.pth"   # saved model weights

device = "cuda" if torch.cuda.is_available() else "cpu"

model = WindowTransformerClassifier(in_ch=10, dim=128, depth=8, heads=4, ws=8, drop=0.1)

state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model = model.to(device)
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


# =========================================================
# 8. Validation report on filtered validation set
# =========================================================
report = full_validation_report(
    model,
    val_loader,
    device=device,
    thresholds=(0.5, 0.75, 0.95),
    use_amp=True
)

print("\n=== FILTERED VALIDATION REPORT ===")
for thr, r in report.items():
    print(
        f"thr={thr}: "
        f"TP={r['tp']} TN={r['tn']} FP={r['fp']} FN={r['fn']} | "
        f"UA={r['UA']:.3f} PA={r['PA']:.3f} OA={r['OA']:.3f} "
        f"AA={r['AA']:.3f} F1={r['F1']:.3f}"
    )

In [ ]:
import os
import pandas as pd
OUT_CSV = "basic_validation_report.csv"

# convert report dict -> table
report_rows = []
for thr, r in report.items():
    row = {"threshold": thr}
    row.update(r)
    report_rows.append(row)

report_df = pd.DataFrame(report_rows)
report_df = report_df[
    ["threshold", "tp", "tn", "fp", "fn", "UA", "PA", "OA", "AA", "F1", "N"]
]

report_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print(report_df)